# 03: Metrics From Scratch

Metrics compress behavior into numbers. That is useful, but also dangerous: a scalar score feels objective even when it only captures one slice of quality. This notebook builds common eval metrics from first principles so the numbers stay interpretable.

## Learning goals

- implement exact match from scratch
- compute overlap-based precision, recall, and F1
- understand micro and macro averaging for classification
- build ranking metrics with simple Python
- see where scalar metrics are helpful and where they fail

In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Why build metrics yourself

Library metrics are convenient, but hidden logic makes it easy to misuse them. Building them once from scratch helps you answer:

- what exactly is being counted
- what gets ignored
- how class imbalance changes the result
- why two metrics can disagree on the same run

## Start with a few toy tasks

We will use three types of examples:

- short question answering pairs
- structured extraction outputs
- ranked retrieval lists

The point is not scale. The point is seeing the mechanics clearly.

In [ ]:

def print_table(rows, headers=None):
    rows = list(rows)
    if not rows:
        print("No rows")
        return

    if headers is None:
        if isinstance(rows[0], dict):
            headers = list(rows[0].keys())
        else:
            headers = [f"col_{index}" for index in range(len(rows[0]))]

    normalized = []
    for row in rows:
        if isinstance(row, dict):
            normalized.append([str(row.get(header, "")) for header in headers])
        else:
            normalized.append([str(value) for value in row])

    widths = []
    for index, header in enumerate(headers):
        content_width = max(len(values[index]) for values in normalized)
        widths.append(max(len(str(header)), content_width))

    header_line = " | ".join(str(header).ljust(widths[index]) for index, header in enumerate(headers))
    divider = "-+-".join("-" * width for width in widths)

    print(header_line)
    print(divider)
    for values in normalized:
        print(" | ".join(values[index].ljust(widths[index]) for index in range(len(headers))))


In [ ]:

qa_examples = [
    {"id": "q1", "gold": "Berlin", "prediction": "Berlin"},
    {"id": "q2", "gold": "Mercury", "prediction": "planet Mercury"},
    {"id": "q3", "gold": "photosynthesis converts sunlight into chemical energy", "prediction": "plants turn sunlight into chemical energy through photosynthesis"},
    {"id": "q4", "gold": "Pacific Ocean", "prediction": "Atlantic Ocean"},
]

print_table(qa_examples, headers=["id", "gold", "prediction"])


## Exact match

Exact match is the strictest simple metric. It gives full credit only when the prediction is exactly equal to the reference. That makes it great for:

- route labels
- IDs
- boolean answers
- rigidly formatted outputs

It is less useful when multiple phrasings are acceptable.

In [ ]:

def exact_match(prediction, gold):
    return int(prediction == gold)

exact_match_rows = []
for example in qa_examples:
    exact_match_rows.append(
        {
            "id": example["id"],
            "prediction": example["prediction"],
            "gold": example["gold"],
            "exact_match": exact_match(example["prediction"], example["gold"]),
        }
    )

print_table(exact_match_rows, headers=["id", "prediction", "gold", "exact_match"])


## Token overlap gives partial credit

When outputs have multiple acceptable phrasings, token overlap can be more informative than exact match. We will normalize text, count overlapping tokens, and compute precision, recall, and F1.

In [ ]:

from collections import Counter

def normalize_text(text):
    characters = []
    for character in text.lower():
        characters.append(character if character.isalnum() else " ")
    return " ".join("".join(characters).split())

def token_counts(text):
    return Counter(normalize_text(text).split())

def overlap_counts(prediction, gold):
    prediction_counts = token_counts(prediction)
    gold_counts = token_counts(gold)
    common = prediction_counts & gold_counts
    true_positive = sum(common.values())
    predicted_total = sum(prediction_counts.values())
    gold_total = sum(gold_counts.values())
    return true_positive, predicted_total, gold_total

def precision_recall_f1(prediction, gold):
    true_positive, predicted_total, gold_total = overlap_counts(prediction, gold)
    precision = true_positive / predicted_total if predicted_total else 0.0
    recall = true_positive / gold_total if gold_total else 0.0
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_positive_tokens": true_positive,
        "predicted_tokens": predicted_total,
        "gold_tokens": gold_total,
    }


In [ ]:

overlap_rows = []
for example in qa_examples:
    scores = precision_recall_f1(example["prediction"], example["gold"])
    overlap_rows.append(
        {
            "id": example["id"],
            "precision": round(scores["precision"], 3),
            "recall": round(scores["recall"], 3),
            "f1": round(scores["f1"], 3),
            "tp_tokens": scores["true_positive_tokens"],
        }
    )

print_table(overlap_rows, headers=["id", "precision", "recall", "f1", "tp_tokens"])


## Precision, recall, and F1 also work for set extraction

Suppose a model must extract required fields from a document. Missing a field hurts recall. Adding fields that are not present hurts precision. F1 balances the two.

In [ ]:

def set_precision_recall_f1(predicted_items, gold_items):
    predicted_set = set(predicted_items)
    gold_set = set(gold_items)
    true_positive = len(predicted_set & gold_set)
    precision = true_positive / len(predicted_set) if predicted_set else 0.0
    recall = true_positive / len(gold_set) if gold_set else 0.0
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1

extraction_examples = [
    {"id": "e1", "gold": ["invoice_number", "due_date", "amount"], "prediction": ["invoice_number", "amount"]},
    {"id": "e2", "gold": ["customer_name", "renewal_date"], "prediction": ["customer_name", "renewal_date", "plan_name"]},
    {"id": "e3", "gold": ["policy_owner", "coverage_limit"], "prediction": ["coverage_limit", "policy_owner"]},
]

extraction_rows = []
for example in extraction_examples:
    precision, recall, f1 = set_precision_recall_f1(example["prediction"], example["gold"])
    extraction_rows.append(
        {
            "id": example["id"],
            "gold_fields": len(example["gold"]),
            "predicted_fields": len(example["prediction"]),
            "precision": round(precision, 3),
            "recall": round(recall, 3),
            "f1": round(f1, 3),
        }
    )

print_table(extraction_rows, headers=["id", "gold_fields", "predicted_fields", "precision", "recall", "f1"])


## Classification metrics come from the confusion counts

For single-label classification, precision and recall can be computed per class from true positives, false positives, and false negatives. From there we can build macro and micro averages.

In [ ]:

classification_examples = [
    {"id": "c1", "gold": "billing", "prediction": "billing"},
    {"id": "c2", "gold": "billing", "prediction": "billing"},
    {"id": "c3", "gold": "billing", "prediction": "account_access"},
    {"id": "c4", "gold": "technical_issue", "prediction": "technical_issue"},
    {"id": "c5", "gold": "technical_issue", "prediction": "billing"},
    {"id": "c6", "gold": "feature_request", "prediction": "feature_request"},
    {"id": "c7", "gold": "feature_request", "prediction": "feature_request"},
    {"id": "c8", "gold": "security", "prediction": "security"},
    {"id": "c9", "gold": "security", "prediction": "account_access"},
    {"id": "c10", "gold": "account_access", "prediction": "account_access"},
]

def per_class_metrics(rows):
    labels = sorted({row["gold"] for row in rows} | {row["prediction"] for row in rows})
    summary = []

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for label in labels:
        tp = sum(1 for row in rows if row["gold"] == label and row["prediction"] == label)
        fp = sum(1 for row in rows if row["gold"] != label and row["prediction"] == label)
        fn = sum(1 for row in rows if row["gold"] == label and row["prediction"] != label)

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

        summary.append(
            {
                "label": label,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "precision": round(precision, 3),
                "recall": round(recall, 3),
                "f1": round(f1, 3),
            }
        )

        total_tp += tp
        total_fp += fp
        total_fn += fn

    macro_precision = sum(row["precision"] for row in summary) / len(summary)
    macro_recall = sum(row["recall"] for row in summary) / len(summary)
    macro_f1 = sum(row["f1"] for row in summary) / len(summary)

    micro_precision = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0.0
    micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if micro_precision + micro_recall else 0.0

    aggregate_rows = [
        {"average": "macro", "precision": round(macro_precision, 3), "recall": round(macro_recall, 3), "f1": round(macro_f1, 3)},
        {"average": "micro", "precision": round(micro_precision, 3), "recall": round(micro_recall, 3), "f1": round(micro_f1, 3)},
    ]

    return summary, aggregate_rows

per_label_rows, aggregate_rows = per_class_metrics(classification_examples)
print("Per-label metrics")
print_table(per_label_rows, headers=["label", "tp", "fp", "fn", "precision", "recall", "f1"])
print()
print("Aggregate metrics")
print_table(aggregate_rows, headers=["average", "precision", "recall", "f1"])


## Macro versus micro tells you what you value

- macro averaging gives each label equal weight
- micro averaging gives each example equal weight

If rare but important classes matter, macro metrics often expose weaknesses that micro metrics smooth over.

In [ ]:

imbalanced_examples = (
    [{"id": f"b{index}", "gold": "billing", "prediction": "billing"} for index in range(1, 16)]
    + [{"id": "s1", "gold": "security", "prediction": "account_access"}]
    + [{"id": "s2", "gold": "security", "prediction": "security"}]
)

imbalanced_per_label, imbalanced_aggregate = per_class_metrics(imbalanced_examples)

print("Imbalanced per-label metrics")
print_table(imbalanced_per_label, headers=["label", "tp", "fp", "fn", "precision", "recall", "f1"])
print()
print("Imbalanced aggregate metrics")
print_table(imbalanced_aggregate, headers=["average", "precision", "recall", "f1"])


## Ranking metrics measure ordering, not just membership

Retrieval systems and rerankers do not only need to include relevant documents. They also need to put them near the top. That is why ranking metrics exist.

In [ ]:

def precision_at_k(relevant_ids, ranked_ids, k):
    ranked_slice = ranked_ids[:k]
    if not ranked_slice:
        return 0.0
    hits = sum(1 for doc_id in ranked_slice if doc_id in relevant_ids)
    return hits / len(ranked_slice)

def recall_at_k(relevant_ids, ranked_ids, k):
    if not relevant_ids:
        return 0.0
    hits = sum(1 for doc_id in ranked_ids[:k] if doc_id in relevant_ids)
    return hits / len(relevant_ids)

def reciprocal_rank(relevant_ids, ranked_ids):
    for index, doc_id in enumerate(ranked_ids, start=1):
        if doc_id in relevant_ids:
            return 1 / index
    return 0.0

def average_precision(relevant_ids, ranked_ids):
    hit_count = 0
    precision_sum = 0.0
    for index, doc_id in enumerate(ranked_ids, start=1):
        if doc_id in relevant_ids:
            hit_count += 1
            precision_sum += hit_count / index
    if not relevant_ids:
        return 0.0
    return precision_sum / len(relevant_ids)

def dcg_at_k(relevance_scores, k):
    total = 0.0
    for index, score in enumerate(relevance_scores[:k], start=1):
        if index == 1:
            total += score
        else:
            total += score / math.log2(index + 1)
    return total

def ndcg_at_k(relevance_by_doc, ranked_ids, k):
    observed_scores = [relevance_by_doc.get(doc_id, 0) for doc_id in ranked_ids[:k]]
    ideal_scores = sorted(relevance_by_doc.values(), reverse=True)[:k]
    ideal_dcg = dcg_at_k(ideal_scores, k)
    if ideal_dcg == 0:
        return 0.0
    return dcg_at_k(observed_scores, k) / ideal_dcg


In [ ]:

retrieval_examples = [
    {
        "query": "refund duplicate renewal charge",
        "relevant": {"d1", "d5"},
        "graded_relevance": {"d1": 3, "d5": 2, "d7": 1},
        "system_a": ["d1", "d2", "d5", "d7", "d8"],
        "system_b": ["d2", "d7", "d5", "d1", "d8"],
    },
    {
        "query": "sso certificate rotation failure",
        "relevant": {"d3", "d6"},
        "graded_relevance": {"d3": 3, "d6": 2, "d4": 1},
        "system_a": ["d6", "d4", "d3", "d1", "d2"],
        "system_b": ["d4", "d2", "d6", "d3", "d5"],
    },
]

ranking_rows = []
for example in retrieval_examples:
    for system_name in ["system_a", "system_b"]:
        ranking = example[system_name]
        ranking_rows.append(
            {
                "query": example["query"],
                "system": system_name,
                "p_at_3": round(precision_at_k(example["relevant"], ranking, 3), 3),
                "recall_at_3": round(recall_at_k(example["relevant"], ranking, 3), 3),
                "mrr": round(reciprocal_rank(example["relevant"], ranking), 3),
                "map": round(average_precision(example["relevant"], ranking), 3),
                "ndcg_at_5": round(ndcg_at_k(example["graded_relevance"], ranking, 5), 3),
            }
        )

print_table(ranking_rows, headers=["query", "system", "p_at_3", "recall_at_3", "mrr", "map", "ndcg_at_5"])


## Compare how metrics react as k changes

Precision at 1 and recall at 5 can tell very different stories. Sweeping k helps you see whether a system retrieves relevant items early or only eventually.

In [ ]:

query_example = retrieval_examples[0]
k_sweep_rows = []
for system_name in ["system_a", "system_b"]:
    ranking = query_example[system_name]
    for k in [1, 2, 3, 5]:
        k_sweep_rows.append(
            {
                "system": system_name,
                "k": k,
                "precision_at_k": round(precision_at_k(query_example["relevant"], ranking, k), 3),
                "recall_at_k": round(recall_at_k(query_example["relevant"], ranking, k), 3),
            }
        )

print_table(k_sweep_rows, headers=["system", "k", "precision_at_k", "recall_at_k"])


## Scalar metrics fail in predictable ways

Metrics are summaries, not reality. Exact match punishes harmless paraphrases. Overlap metrics can reward long but sloppy answers. Ranking metrics say nothing about whether retrieved documents are actually used correctly downstream.

In [ ]:

failure_cases = [
    {
        "case": "Paraphrase punished by exact match",
        "gold": "photosynthesis converts sunlight into chemical energy",
        "prediction": "plants turn sunlight into chemical energy through photosynthesis",
    },
    {
        "case": "Verbose answer earns overlap credit",
        "gold": "Berlin",
        "prediction": "Berlin is the capital of Germany and a major European city with a long history",
    },
    {
        "case": "Wrong entity with some shared words",
        "gold": "Pacific Ocean",
        "prediction": "Atlantic Ocean",
    },
]

failure_rows = []
for row in failure_cases:
    overlap = precision_recall_f1(row["prediction"], row["gold"])
    failure_rows.append(
        {
            "case": row["case"],
            "exact_match": exact_match(normalize_text(row["prediction"]), normalize_text(row["gold"])),
            "overlap_f1": round(overlap["f1"], 3),
        }
    )

print_table(failure_rows, headers=["case", "exact_match", "overlap_f1"])


## A small scorer registry makes experiments easier to extend

Once the metric functions exist, you can register them and run multiple scorers over the same predictions. This is the start of a reusable eval harness.

In [ ]:

SCORERS = {
    "exact_match_normalized": lambda prediction, gold: exact_match(normalize_text(prediction), normalize_text(gold)),
    "token_f1": lambda prediction, gold: round(precision_recall_f1(prediction, gold)["f1"], 3),
    "token_precision": lambda prediction, gold: round(precision_recall_f1(prediction, gold)["precision"], 3),
    "token_recall": lambda prediction, gold: round(precision_recall_f1(prediction, gold)["recall"], 3),
}

registry_rows = []
for example in qa_examples:
    row = {"id": example["id"]}
    for scorer_name, scorer in SCORERS.items():
        row[scorer_name] = scorer(example["prediction"], example["gold"])
    registry_rows.append(row)

print_table(
    registry_rows,
    headers=["id", "exact_match_normalized", "token_f1", "token_precision", "token_recall"],
)


## Takeaways

- exact match is powerful when the target space is rigid
- precision, recall, and F1 reveal different failure modes
- macro and micro averages answer different questions
- ranking metrics care about order, not only inclusion
- every metric hides some part of reality, so metric choice is part of benchmark design

Next we will move from scoring into diagnosis: once a run fails, how do you organize those failures into useful buckets and roadmap decisions?